In [ ]:
!pip install birdhouse-birdy

In [10]:
# read config file first
import json
with open('config.json', 'r') as f:
    config = json.load(f)
print(config)

{u'COP_user': u'', u'PEPS_formats': [u'json', u'atom'], u'PEPS_url': u'https://peps.cnes.fr/resto/api/collections/', u'COP_pwd': u'', u'COP_url': u'https://scihub.copernicus.eu/dhus/search?q=', u'PAVICS_url': u'https://pavics.ouranos.ca/twitcher/ows/proxy/catalog/wps', u'PEPS_collections': [u'S1', u'S2', u'S2ST', u'S3']}


# Connecteur API PEPS

Reference: https://peps.cnes.fr/rocket/plus/img/PEPS-IF-0-0170-ATOS_01_00_[2].pdf
 (pages 8-11)

Examples:
"recherche de produits **S2ST** avec un identifiant **MGRS** égal à **31TCJ** et avec une **couverture nuageuse** comprise entre **10% et 70%**":

https://peps.cnes.fr/resto/api/collections/S2ST/search.json?tileid=31TCJ&cloudCover=[10,70]


La liste des critères de recherche et valeurs disponibles pour chacun des critères est disponible aux adresses
suivantes :
- Les descripteurs Opensearch de la collection [Sentinel 1](https://peps.cnes.fr/resto/api/collections/S1/describe.xml)
- Les descripteurs Opensearch de la collection [Sentinel 2](https://peps.cnes.fr/resto/api/collections/S2/describe.xml)
- Les descripteurs Opensearch de la collection [Sentinel 2 tuilés](https://peps.cnes.fr/resto/api/collections/S2ST/describe.xml)
- Les descripteurs Opensearch de la collection [Sentinel 3](https://peps.cnes.fr/resto/api/collections/S3/describe.xml)


In [11]:
parameters = {
    'tileid' : "31TCJ",
    'cloudCover' : '[10,70]'
}
collection = "S2ST"
out_format = "json"

In [12]:
# construct the search URL
def PEPS_search_url(parameters, collection, out_format):
    search_URL = config['PEPS_url']
    if collection in config['PEPS_collections'] :
      search_URL += collection +"/"
    if out_format in config['PEPS_formats']:
      search_URL += "search." + out_format
    # add any parameters
    for key in parameters:
      search_URL += "&" + key + "=" + str(parameters[key])
    # replace first & with ?
    search_URL = search_URL.replace("&", "?", 1)
    return search_URL

In [13]:
search_PEPS = PEPS_search_url(parameters, collection, out_format)
print(search_PEPS)

https://peps.cnes.fr/resto/api/collections/S2ST/search.json?tileid=31TCJ&cloudCover=[10,70]


Save the query result as a json file

In [14]:
import requests
r = requests.get(search_PEPS, allow_redirects=True)
open('out_peps.json', 'wb').write(r.content)

In [15]:
!ls

config.json  DACCS_connecteurs_externes.ipynb  out_peps.json  Untitled.ipynb


Open the search result

In [16]:
with open ('out_peps.json', 'r') as f:
  out_f = json.load(f)
out_f

{u'features': [{u'_geometry': {u'coordinates': [[[[0.53677232313667,
        43.238262553327],
       [1.8886830141923, 43.259391910537],
       [1.8702372867615, 44.247830683969],
       [0.49592859290379, 44.225964154763],
       [0.53677232313667, 43.238262553327]]]],
    u'type': u'MultiPolygon'},
   u'geometry': {u'coordinates': [[[[0.53677232313667, 43.238262553327],
       [1.8886830141923, 43.259391910537],
       [1.8702372867615, 44.247830683969],
       [0.49592859290379, 44.225964154763],
       [0.53677232313667, 43.238262553327]]]],
    u'type': u'MultiPolygon'},
   u'id': u'fc8e33da-5c5c-5651-885b-53015e9bb371',
   u'properties': {u'bareSoil': None,
    u'cloudCover': 16.9528,
    u'collection': u'S2ST',
    u'completionDate': u'2020-09-12T10:46:29.024Z',
    u'description': None,
    u'dhusIngestDate': u'2020-09-12T15:30:59.236Z',
    u'highProbaClouds': None,
    u'instrument': u'MSI',
    u'isNrt': 0,
    u'keywords': {u'44b15c0a6e62129': {u'href': u'https://peps.cnes

In [17]:
out_f['properties']['totalResults']

208

# Connecteur COPERNICUS
Description: https://scihub.copernicus.eu/userguide/OpenSearchAPI

In [20]:
q = "footprint:\"Intersects(41.9000, 12.5000)\""
search_COP = config['COP_url'] + q
search_COP

u'https://scihub.copernicus.eu/dhus/search?q=footprint:"Intersects(41.9000, 12.5000)"'

In [22]:
import requests
r = requests.get(search_COP, allow_redirects=True, auth=(config['COP_user'], config['COP_pwd']))
open('out_cop.xml', 'wb').write(r.content)

In [23]:
!ls

config.json			  out_cop.xml	 Untitled.ipynb
DACCS_connecteurs_externes.ipynb  out_peps.json


In [24]:
import xml.etree.ElementTree as ET
tree = ET.parse('out_cop.xml')
root = tree.getroot()
for child in root:
    print(child.tag, child.attrib, child.text)
    for gchild in child:
          print("\t", gchild.tag, gchild.attrib, gchild.text)

('{http://www.w3.org/2005/Atom}title', {}, 'Sentinels Scientific Data Hub search results for: footprint:"Intersects(41.9000, 12.5000)"')
('{http://www.w3.org/2005/Atom}subtitle', {}, 'Displaying 0 to 9 of 24117 total results. Request done in 0.012 seconds.')
('{http://www.w3.org/2005/Atom}updated', {}, '2020-09-18T15:05:27.280Z')
('{http://www.w3.org/2005/Atom}author', {}, '\n')
('\t', '{http://www.w3.org/2005/Atom}name', {}, 'Sentinels Scientific Data Hub')
('{http://www.w3.org/2005/Atom}id', {}, 'https://scihub.copernicus.eu/dhus/search?q=footprint:"Intersects(41.9000, 12.5000)"')
('{http://a9.com/-/spec/opensearch/1.1/}totalResults', {}, '24117')
('{http://a9.com/-/spec/opensearch/1.1/}startIndex', {}, '0')
('{http://a9.com/-/spec/opensearch/1.1/}itemsPerPage', {}, '10')
('{http://a9.com/-/spec/opensearch/1.1/}Query', {'role': 'request', 'startPage': '1', 'searchTerms': 'footprint:"Intersects(41.9000, 12.5000)"'}, None)
('{http://www.w3.org/2005/Atom}link', {'href': 'https://scihub.

# Connecteur PAVICS


In [ ]:
from birdy import WPSClient
wps = WPSClient(config['PAVICS_url'])
help(wps.pavicsearch)

In [ ]:
params = "variable:tasmin,project:CMIP5,experiment:rcp85,frequency:day,institute:CCCma,model:CanESM2"

In [ ]:
resp = wps.pavicsearch(constraints=params, limit=10, type="File")
[result, files] = resp.get(asobj=True)
files